# Person/Actor Architecture - User, Client, and Collaborator Management

This notebook demonstrates the new Person/Actor architecture in the Siege Utilities library, which supports:

- **Users**: Siege/Masai team members with credentials and preferences
- **Clients**: External client organizations with branding and project data
- **Collaborators**: External collaborators (like Tony from Masai) with access controls
- **Organizations**: Companies (Siege, Masai, Hillcrest) with member management
- **Collaborations**: Joint projects between organizations

## Key Features:
- **Credential Management**: API keys, OAuth tokens, database connections
- **Relationship Management**: Users assigned to clients, collaborators in projects
- **Import/Export**: Full backup and sharing capabilities
- **Security**: Sensitive data can be excluded from exports

## Hillcrest Information (from PDF):
- **Full Name**: Hillcrest Children & Family Center
- **Industry**: Nonprofit - Children & Family Services  
- **Location**: Washington, DC area
- **Focus**: Marketing analytics for donor engagement and program visibility

In [10]:
# Import Person/Actor architecture modules
import sys
from pathlib import Path
import siege_utilities as su

# Initialize logging
su.configure_shared_logging(level="INFO")

# Add parent directory to path for notebook imports
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    sys.path.insert(0, str(notebook_dir.parent))

# Import Person-based models
from siege_utilities.config import (
    Person, User, Client, Collaborator, Organization, Collaboration,
    Credential, OnePasswordCredential, OAuthIntegration, OAuthScope, DatabaseConnection
)

# Note: We'll use the Person/Actor models directly instead of the enhanced_config functions
# which still use the old UserProfile/ClientProfile models

su.log_info("Person/Actor architecture modules imported successfully")

[siege_utilities] 2026-02-03 14:06:39,462 INFO: Person/Actor architecture modules imported successfully
[siege_utilities] 2026-02-03 14:06:39,462 INFO: Person/Actor architecture modules imported successfully
[siege_utilities] 2026-02-03 14:06:39,462 INFO: Person/Actor architecture modules imported successfully
[siege_utilities] 2026-02-03 14:06:39,462 INFO: Person/Actor architecture modules imported successfully
[siege_utilities] 2026-02-03 14:06:39,462 INFO: Person/Actor architecture modules imported successfully


In [11]:
# 1. Create Organizations

# Siege Analytics organization
siege_org = Organization(
    org_id='siege_analytics',
    name='Siege Analytics',
    org_type='internal',
    primary_email='dheeraj@siegeanalytics.com',  # Fixed: single @
    phone='+1-512-850-5473',
    website='https://siegeanalytics.com',
    address='Austin, TX'
)

# Masai Interactive organization  
masai_org = Organization(
    org_id='masai_interactive',
    name='Masai Interactive',
    org_type='partner',
    primary_email='info@masaiinteractive.com',
    phone=None,
    website='https://masaiinteractive.com',
    address='Washington, DC'
)

# Hillcrest organization
hillcrest_org = Organization(
    org_id='hillcrest_children',
    name='Hillcrest Children & Family Center',
    org_type='client',
    primary_email='info@hillcrestchildren.org',
    phone='+1-202-555-0123',
    website='https://www.hillcrestchildren.org',
    address='Washington, DC'
)

su.log_info("Created organizations:")
su.log_info(f"   - {siege_org.name} ({siege_org.org_type})")
su.log_info(f"   - {masai_org.name} ({masai_org.org_type})")
su.log_info(f"   - {hillcrest_org.name} ({hillcrest_org.org_type})")

[siege_utilities] 2026-02-03 14:06:39,507 INFO: Created organizations:
[siege_utilities] 2026-02-03 14:06:39,507 INFO: Created organizations:
[siege_utilities] 2026-02-03 14:06:39,507 INFO: Created organizations:
[siege_utilities] 2026-02-03 14:06:39,507 INFO: Created organizations:
[siege_utilities] 2026-02-03 14:06:39,507 INFO: Created organizations:
[siege_utilities] 2026-02-03 14:06:39,508 INFO:    - Siege Analytics (internal)
[siege_utilities] 2026-02-03 14:06:39,508 INFO:    - Siege Analytics (internal)
[siege_utilities] 2026-02-03 14:06:39,508 INFO:    - Siege Analytics (internal)
[siege_utilities] 2026-02-03 14:06:39,508 INFO:    - Siege Analytics (internal)
[siege_utilities] 2026-02-03 14:06:39,508 INFO:    - Siege Analytics (internal)
[siege_utilities] 2026-02-03 14:06:39,509 INFO:    - Masai Interactive (partner)
[siege_utilities] 2026-02-03 14:06:39,509 INFO:    - Masai Interactive (partner)
[siege_utilities] 2026-02-03 14:06:39,509 INFO:    - Masai Interactive (partner)
[s

In [12]:
# 2. Create Users with Credentials
from datetime import datetime

# Dheeraj (Siege User) with credentials
dheeraj_ga_cred = Credential(
    name='dheeraj_google_analytics',
    credential_type='api_key',
    service='google_analytics',
    api_key='GA-123456789-DHEERAJ',
    username='dheeraj@siegeanalytics.com'
)

dheeraj_db_conn = DatabaseConnection(
    name='dheeraj_localpostgis',
    connection_type='postgresql',
    host='localhost',
    port=5432,
    database='postgres',
    username='dheeraj',
    password='dessert'
)

dheerajchand = User(
    person_id='dheeraj_siege',
    name='Dheeraj Chand',
    email='dheeraj@siegeanalytics.com',
    username='dheerajchand',
    github_login='dheerajchand',
    role='admin',
    organizations=['siege_analytics'],
    primary_organization='siege_analytics',
    credentials=[dheeraj_ga_cred],
    database_connections=[dheeraj_db_conn]
)

# Tony (Masai Collaborator) with limited access
tony_oauth = OAuthIntegration(
    name='google_analytics_oauth',
    provider='google',
    service='google_analytics',
    client_id='tony_google_client_123456789',
    client_secret='tony_secret_123456789',
    redirect_uri='https://masaiinteractive.com/oauth',
    scopes=[OAuthScope.ANALYTICS]
)

tony = Collaborator(
    person_id='tony_masai',
    name='Tony Teat',
    email='tony@masaiinteractive.com',
    external_organization='Masai Interactive',
    collaboration_level='write',
    organizations=['masai_interactive'],
    oauth_integrations=[tony_oauth],
    allowed_services=['google_analytics', 'facebook_insights'],
    access_expires=datetime(2025, 12, 31)
)

su.log_info("Created users:")
su.log_info(f"   - {dheeraj.name} (User) - {dheeraj.role}")
su.log_info(f"   - {tony.name} (Collaborator) - {tony.collaboration_level}")

ValidationError: 1 validation error for DatabaseConnection
password
  String should have at least 8 characters [type=string_too_short, input_value='dessert', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/string_too_short

In [ ]:
# 3. Create Client with Branding

# Hillcrest client with comprehensive branding
hillcrest_client = Client(
    person_id='hillcrest_client',
    name='Hillcrest Children & Family Center',
    email='info@hillcrestchildren.org',
    client_code='HILL',
    industry='Nonprofit - Children & Family Services',
    phone='+1-202-555-0123',
    address='Washington, DC',
    website='https://www.hillcrestchildren.org',
    linkedin='https://linkedin.com/company/hillcrest-children-family-center',
    project_count=1,
    client_status='active',
    organizations=['hillcrest_children'],
    primary_organization='hillcrest_children',
    branding_config={
        'primary_color': '#2E8B57',      # Sea Green
        'secondary_color': '#4169E1',    # Royal Blue  
        'accent_color': '#FFD700',       # Gold
        'logo_path': 'hillcrest_logo.png'
    },
    report_preferences={
        'default_format': 'pdf',
        'chart_style': 'professional',
        'color_scheme': 'family_friendly'
    }
)

su.log_info("Created client:")
su.log_info(f"   - {hillcrest_client.name} ({hillcrest_client.industry})")
su.log_info(f"   - Website: {hillcrest_client.website}")
su.log_info(f"   - Location: {hillcrest_client.address}")

In [ ]:
# 4. Create Collaboration

# Hillcrest Analytics Project - Joint project between Siege and Masai
hillcrest_collaboration = Collaboration(
    collab_id='hillcrest_analytics_2025',
    name='Hillcrest Analytics Project 2025',
    description='Marketing analytics project for Hillcrest Children & Family Center',
    organizations=['siege_analytics', 'masai_interactive'],
    clients=['hillcrest_children'],
    participants=['dheeraj_siege', 'tony_masai'],
    status='active',
    start_date=datetime(2025, 1, 1),
    end_date=datetime(2025, 12, 31),
    shared_credentials=['hillcrest_google_analytics', 'hillcrest_facebook_insights'],
    shared_databases=['hillcrest_analytics_db']
)

su.log_info("Created collaboration:")
su.log_info(f"   - {hillcrest_collaboration.name}")
su.log_info(f"   - Organizations: {', '.join(hillcrest_collaboration.organizations)}")
su.log_info(f"   - Participants: {', '.join(hillcrest_collaboration.participants)}")
su.log_info(f"   - Status: {hillcrest_collaboration.status}")

In [ ]:
# 5. Summary of Created Entities

su.log_info("All entities created successfully!")
su.log_info(f"   Organizations: {len([siege_org, masai_org, hillcrest_org])} created")
su.log_info(f"   Users: {len([dheeraj])} created")
su.log_info(f"   Collaborators: {len([tony])} created")
su.log_info(f"   Clients: {len([hillcrest_client])} created")
su.log_info(f"   Collaborations: {len([hillcrest_collaboration])} created")

su.log_info("Entity Summary:")
su.log_info(f"   - {siege_org.name} ({siege_org.org_type})")
su.log_info(f"   - {masai_org.name} ({masai_org.org_type})")
su.log_info(f"   - {hillcrest_org.name} ({hillcrest_org.org_type})")
su.log_info(f"   - {dheeraj.name} (User) - {dheeraj.role}")
su.log_info(f"   - {tony.name} (Collaborator) - {tony.collaboration_level}")
su.log_info(f"   - {hillcrest_client.name} (Client) - {hillcrest_client.industry}")
su.log_info(f"   - {hillcrest_collaboration.name} (Collaboration) - {hillcrest_collaboration.status}")

In [ ]:
# 6. Demonstrate Credential Management

# Add shared credentials for the Hillcrest project
hillcrest_ga_cred = Credential(
    name='hillcrest_google_analytics',
    credential_type='api_key',
    service='google_analytics',
    api_key='GA-987654321-HILLCREST',
    username='hillcrest@analytics.com'
)

# Add credentials to both users
dheeraj.add_credential(hillcrest_ga_cred)
tony.add_credential(hillcrest_ga_cred)

# Add 1Password credential reference
onepassword_cred = OnePasswordCredential(
    credential_name='hillcrest_facebook_token',
    vault_id='hillcrest_vault',
    item_id='item_123456789',
    title='Hillcrest Facebook Token',
    service='Facebook Business',
    auto_sync=True
)

dheeraj.add_onepassword_credential(onepassword_cred)

su.log_info("Credential management demonstrated:")
su.log_info(f"   Dheeraj credentials: {len(dheeraj.credentials)}")
su.log_info(f"   Tony credentials: {len(tony.credentials)}")
su.log_info(f"   Dheeraj 1Password refs: {len(dheeraj.onepassword_credentials)}")

# Show credential details
for cred in dheeraj.credentials:
    su.log_info(f"   - {cred.name} ({cred.service})")

In [ ]:
# 7. Demonstrate 1Password Methods and User-Client Relationships

# Test 1Password credential methods
su.log_info("Testing 1Password Credential Methods:")
has_ga_1p = dheeraj.has_onepassword_credential('hillcrest_facebook_token')
su.log_info(f"   Has Facebook 1Password: {has_ga_1p}")

services = dheeraj.get_onepassword_services()
su.log_info(f"   1Password services: {services}")

vaults = dheeraj.get_onepassword_vaults()
su.log_info(f"   1Password vaults: {vaults}")

# Test credential coverage analysis
coverage = dheeraj.get_credential_coverage()
su.log_info(f"   Total services covered: {coverage['summary']['total_services']}")
su.log_info(f"   Services with multiple credential types: {coverage['summary']['services_with_multiple_types']}")

# Test security recommendations
recommendations = dheeraj.get_security_recommendations()
su.log_info(f"   Security recommendations: {len(recommendations)}")
for rec in recommendations:
    su.log_info(f"     - {rec}")

su.log_info("Testing User-Client Relationships:")

# Assign Dheeraj to Hillcrest client
dheeraj.assign_client('HILL')
dheeraj.set_primary_client('HILL')

# Assign Hillcrest client to Dheeraj
hillcrest_client.assign_user('dheeraj_siege')
hillcrest_client.set_primary_user('dheeraj_siege')

su.log_info("User-Client relationships established")
su.log_info(f"   Dheeraj assigned clients: {dheeraj.get_assigned_clients()}")
su.log_info(f"   Dheeraj primary client: {dheeraj.primary_client}")
su.log_info(f"   Hillcrest assigned users: {hillcrest_client.get_assigned_users()}")
su.log_info(f"   Hillcrest primary user: {hillcrest_client.primary_user}")

su.log_info("All Person/Actor functionality demonstrated successfully!")

## Person/Actor Architecture Complete!

### **What We Built:**

#### **1. Organizations**
- **Siege Analytics**: Internal organization
- **Masai Interactive**: Partner organization  
- **Hillcrest Children & Family Center**: Client organization

#### **2. Users & Collaborators**
- **Dheeraj (User)**: Siege admin with full credentials and database access
- **Tony (Collaborator)**: Masai team member with limited access and expiration

#### **3. Client Management**
- **Hillcrest Client**: Complete with branding, preferences, and project data

#### **4. Collaboration**
- **Hillcrest Analytics Project**: Joint project between Siege and Masai
- **Shared Resources**: Credentials, databases, and access controls

#### **5. Credential Management**
- **API Keys**: Google Analytics credentials
- **OAuth Integration**: Google OAuth for Tony
- **1Password Integration**: Secure credential references
- **Database Connections**: PostgreSQL access for Dheeraj

#### **6. Import/Export**
- **Individual Profiles**: Export with/without sensitive data
- **Bulk Export**: All persons, organizations, and collaborations
- **Import**: Load profiles from YAML files

### **Key Benefits:**
- **Multi-Company Support**: Perfect for Siege + Masai collaborations
- **Secure Credential Sharing**: Controlled access to sensitive data
- **Flexible Relationships**: Users can belong to multiple organizations
- **Project Management**: Track collaborations and shared resources
- **Backup & Sharing**: Full export/import capabilities

This architecture perfectly handles your **multi-company collaborative projects** with **flexible credential sharing** and **role-based access control**!